# 放心借 lookalike — 训练数据导出（云分析机 / dtools）

前置：集群已跑通 `model/sql/build_pu_training_table.sql`，表 `fxj_lookalike_pu_training` 存在。

本地训练默认读：`model/data/training_pu.parquet`（见 `config.yaml`）。

**注意**：全量约千万行 × 极宽列，首次建议开 `DRY_RUN` 或提高 `UNLABELED_SAMPLE_FRAC` 试跑；CSV 仅适合小样。

In [ ]:
import os

import dtools

# ---------- 表名（与你们控制台一致：两段式或三段式二选一）----------
TABLE_NAME = "ai_decision_dev.fxj_lookalike_pu_training"
# TABLE_NAME = "lj_iceberg.ai_decision_dev.fxj_lookalike_pu_training"

# ---------- 输出路径（相对云分析机工作目录，按实际改）----------
OUTPUT_PARQUET = "model/data/training_pu.parquet"
OUTPUT_CSV = None  # 例如 "model/data/training_pu_sample.csv"，全量勿用 CSV

# ---------- 过滤与抽样 ----------
SPLITS = ("train", "val")
REQUIRE_MS13 = False  # True 时仅保留 ms13_score 非空（需先跑 attach_ms13 SQL）
MS13_MIN_FOR_UNLABELED = 0  # 例如 630：仅对 pu_label=0 生效；0 表示不筛
# 背景行过多时，导出阶段先随机抽未标注（训练脚本里还会 unlabeled_subsample）
UNLABELED_SAMPLE_FRAC = 1.0  # (0,1]，试跑可设 0.05
DRY_RUN_LIMIT = None  # 试跑可设 100000

META_FOR_TRAIN = ["unique_id", "user_id", "pu_label", "dataset_split"]
OPTIONAL_META = ["ms13_score", "label"]

In [ ]:
split_list = ", ".join(repr(s) for s in SPLITS)

ms13_clause = ""
if REQUIRE_MS13:
    ms13_clause += "\n  AND ms13_score IS NOT NULL"
if MS13_MIN_FOR_UNLABELED and MS13_MIN_FOR_UNLABELED > 0:
    ms13_clause += (
        f"\n  AND (pu_label = 1 OR ms13_score >= {float(MS13_MIN_FOR_UNLABELED)})"
    )

sample_clause = ""
if UNLABELED_SAMPLE_FRAC < 1.0:
    sample_clause = (
        f"\n  AND (pu_label = 1 OR rand() < {float(UNLABELED_SAMPLE_FRAC)})"
    )

limit_clause = f"\nLIMIT {int(DRY_RUN_LIMIT)}" if DRY_RUN_LIMIT else ""

# 宽表列极多：用 SELECT *，本地 train.py 通过 features.py 自动剔 id/label 泄漏列
query = f"""
SELECT *
FROM {TABLE_NAME}
WHERE dataset_split IN ({split_list})
{ms13_clause}
{sample_clause}
{limit_clause}
""".strip()

print("SQL 预览:\n", query)

In [ ]:
df_data = dtools.get_as_frame(query)
print(f"行数: {len(df_data)}, 列数: {len(df_data.columns)}")

if "pu_label" in df_data.columns:
    print(df_data["pu_label"].value_counts().sort_index())
if "dataset_split" in df_data.columns and "pu_label" in df_data.columns:
    print(
        df_data.groupby(["pu_label", "dataset_split"], dropna=False)
        .size()
        .reset_index(name="cnt")
    )

missing_meta = [c for c in META_FOR_TRAIN if c not in df_data.columns]
if missing_meta:
    raise ValueError(f"缺少训练元数据列: {missing_meta}")

In [ ]:
out_dir = os.path.dirname(OUTPUT_PARQUET) or "."
os.makedirs(out_dir, exist_ok=True)

df_data.to_parquet(OUTPUT_PARQUET, index=False)
print("已保存", OUTPUT_PARQUET)

if OUTPUT_CSV:
    os.makedirs(os.path.dirname(OUTPUT_CSV) or ".", exist_ok=True)
    df_data.to_csv(OUTPUT_CSV, index=False)
    print("已保存", OUTPUT_CSV)

---
## 附录：征信 + 马消特征表（`jcr_credit_feature_label_full`）

若仍从同事加工的 **jcr** 全量特征表导出（与放心借宽表 PU 流程不同），用下面单元格；列清单与原先脚本一致。

In [ ]:
JCR_TABLE = "lj_iceberg.ai_decision_dev.jcr_credit_feature_label_full_20260715"
JCR_DATA_FILE = "model/data/training_full.csv"

MX_WINS = ("1w", "1m", "3m", "6m", "1y")
MX_PREFIXES = (
    "mx_new_loan_cnt",
    "mx_overdue_loan_cnt",
    "mx_overdue_avg_terms",
    "mx_overdue_max_terms",
    "mx_overdue_amt",
    "mx_settle_loan_cnt",
    "mx_prepay_loan_cnt",
)
JCR_EXPORT_COLUMNS = [
    "latest_pos_bal_acct_cnt",
    "avg_1m_pos_bal_acct_cnt",
    "avg_6m_pos_bal_acct_cnt",
    "avg_1y_pos_bal_acct_cnt",
    "latest_bal_sum",
    "latest_bal_max",
    "latest_bal_min",
    "avg_1m_bal_sum",
    "avg_1m_bal_max",
    "avg_1m_bal_min",
    "avg_6m_bal_sum",
    "avg_6m_bal_max",
    "avg_6m_bal_min",
    "avg_1y_bal_sum",
    "avg_1y_bal_max",
    "avg_1y_bal_min",
    "latest_crdt_sum",
    "latest_crdt_max",
    "latest_crdt_min",
    "avg_1m_crdt_sum",
    "avg_1m_crdt_max",
    "avg_1m_crdt_min",
    "avg_6m_crdt_sum",
    "avg_6m_crdt_max",
    "avg_6m_crdt_min",
    "avg_1y_crdt_sum",
    "avg_1y_crdt_max",
    "avg_1y_crdt_min",
    "latest_util_sum",
    "latest_util_max",
    "latest_util_min",
    "avg_1m_util_sum",
    "avg_1m_util_max",
    "avg_1m_util_min",
    "avg_6m_util_sum",
    "avg_6m_util_max",
    "avg_6m_util_min",
    "avg_1y_util_sum",
    "avg_1y_util_max",
    "avg_1y_util_min",
    "latest_bill_day_cnt",
    "latest_same_billday_acct_cnt_max",
    "latest_same_billday_bal_sum_max",
    "max_1m_same_billday_bal_sum",
    "max_6m_same_billday_bal_sum",
    "max_1y_same_billday_bal_sum",
    "latest_credit_audit_query_org_num_1m",
    "avg_1y_credit_audit_query_org_num_1m",
    "latest_loan_audit_query_num_1m",
    "avg_1y_loan_audit_query_num_1m",
    "latest_credit_audit_query_num_1m",
    "avg_1y_credit_audit_query_num_1m",
    "latest_person_query_num_1m",
    "avg_1y_person_query_num_1m",
    "latest_plm_query_num_2y",
    "avg_1y_plm_query_num_2y",
    "latest_assure_query_num_2y",
    "avg_1y_assure_query_num_2y",
    "latest_sam_query_num_2y",
    "avg_1y_sam_query_num_2y",
    "latest_pd_num_month",
    "latest_pd_total_overdue_cnt",
    "latest_pd_max_overdue_months",
    "latest_pd_max_overdue_amt",
    "latest_has_house_loan_flg",
    "latest_has_gjj_loan_flg",
    "latest_credit_account_num",
    "latest_credit_amount",
    "latest_credit_used_amount",
    "latest_credit_util_rate",
    "latest_org_type",
    "pril_bal",
    "pril_bal_1w",
    "pril_bal_1m",
    "pril_bal_3m",
    "pril_bal_6m",
    "pril_bal_1y",
    "crdt_lim_yx",
    "crdt_lim_yx_1w",
    "crdt_lim_yx_1m",
    "crdt_lim_yx_3m",
    "crdt_lim_yx_6m",
    "crdt_lim_yx_1y",
    "pril_bal_rate",
    "pril_bal_rate_1w",
    "pril_bal_rate_1m",
    "pril_bal_rate_3m",
    "pril_bal_rate_6m",
    "pril_bal_rate_1y",
    "fq_cnt_1w",
    "fq_cnt_1m",
    "fq_cnt_3m",
    "fq_cnt_6m",
    "fq_cnt_1y",
    "suc_cnt_1w",
    "suc_cnt_1m",
    "suc_cnt_3m",
    "suc_cnt_6m",
    "suc_cnt_1y",
] + [f"{p}_{w}" for p in MX_PREFIXES for w in MX_WINS] + [
    "mx_age",
    "mx_job",
    "mx_city_level",
    "mx_education",
    "mx_monthly_income",
    "mx_other_income",
    "mx_response_score",
    "mx_risk_score",
    "zx_balance_label",
    "dataset_split",
    "label_eligible",
    "m",
]

jcr_cols = ",\n    ".join(JCR_EXPORT_COLUMNS)
jcr_query = f"""
SELECT
    {jcr_cols}
FROM {JCR_TABLE}
WHERE label_eligible = 1
  AND zx_balance_label IS NOT NULL
  AND dataset_split IN ('train', 'val')
  AND m IN ('202508', '202509', '202510')
"""

df_jcr = dtools.get_as_frame(jcr_query)
print(f"行数: {len(df_jcr)}, 列数: {len(df_jcr.columns)}")

os.makedirs(os.path.dirname(JCR_DATA_FILE) or ".", exist_ok=True)
df_jcr.to_csv(JCR_DATA_FILE, index=False, header=True)
print("已保存", JCR_DATA_FILE)